In [ ]:
# FEATURE_CROSS_YEAR_STABILITY
# LOOKBACK_168_DIAGNOSTICS

import pandas as pd
import numpy as np
import warnings
import os, sys
import pickle

from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.stats.diagnostic import acorr_ljungbox
from scipy.stats import jarque_bera

In [ ]:
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)


sys.path.insert(0,"/Users/shah/CODE_BOOK_4/THESIS_WORKING/THESIS_2")
sys.path.insert(0,"/Users/shah/CODE_BOOK_4/code_document/REPOSITORIES/MFIV_calibration")

os.chdir("/Users/shah/CODE_BOOK_4/THESIS_WORKING/THESIS_2")

In [ ]:
# =============================================================================================================
# ======================================= LOAD BASE DATA ======================================================
# =============================================================================================================

with open("/Users/shah/CODE_BOOK_4/code_document/REPOSITORIES/data/IS/outputs_by_year/outputs_by_year_03.pkl","rb") as f:
    outputs_by_year = pickle.load(f)

with open("/Users/shah/CODE_BOOK_4/code_document/REPOSITORIES/data/IS/x_by_year_fut/x_by_year.pkl","rb") as f:
    X_by_year = pd.read_pickle(f)


YEARS = [2019, 2020, 2021]
QUARTERS = ["q1", "q2", "q3", "q4"]

In [ ]:
# =============================================================================================================
# ======================================= PREPARE X_BY_YEAR ===================================================
# =============================================================================================================

for y in YEARS:
    X_by_year[y] = X_by_year[y].copy()
    X_by_year[y]["rv"] = X_by_year[y]["rt"] ** 2


Xq_by_year = {}

for y in YEARS:
    Xq_by_year[y] = {}
    Xq_by_year[y]["q1"] = X_by_year[y].loc[f"{y}-01-01" : f"{y}-03-31 23:59:59"]
    Xq_by_year[y]["q2"] = X_by_year[y].loc[f"{y}-04-01" : f"{y}-06-30 23:59:59"]
    Xq_by_year[y]["q3"] = X_by_year[y].loc[f"{y}-07-01" : f"{y}-09-30 23:59:59"]
    Xq_by_year[y]["q4"] = X_by_year[y].loc[f"{y}-10-01" : f"{y}-12-31 23:59:59"]

In [ ]:
# =============================================================================================================
# ======================================= MFIV SCALING ========================================================
# =============================================================================================================
# CONVERTS ANNUALIZED MFIV -> CUMULATIVE H-HOUR DECIMAL VARIANCE

MFIVS = list(range(2, 577))

for y in YEARS:
    for q in QUARTERS:
        for H in MFIVS:
            Xq_by_year[y][q][f"MFIV_{H}"] *= (H / 8760.0)

for y in YEARS:
    for q in QUARTERS:
        Xq_by_year[y][q] = Xq_by_year[y][q].loc[:, ~Xq_by_year[y][q].columns.duplicated(keep="first")]

In [ ]:
# =============================================================================================================
# ======================================= BUILD COMMON MODEL FEATURES ========================================
# =============================================================================================================

MFIV_HORIZONS = [12, 24, 72, 168, 336]
RV_HORIZONS = [12, 24, 72]

features_by_year = {}

for y in YEARS:
    X_df = pd.concat([Xq_by_year[y][q] for q in QUARTERS]).sort_index().copy()
    garch_df = pd.concat([outputs_by_year[y][q]["garch"] for q in QUARTERS]).sort_index()

    X_df.index = pd.to_datetime(X_df.index, utc=True).floor("h")
    X_df = X_df.loc[~X_df.index.duplicated(keep="first")]
    X_df["garch"] = garch_df.reindex(X_df.index)

    features = pd.DataFrame(index=X_df.index)

    for H in MFIV_HORIZONS:
        features[f"MFIV_{H}"] = X_df[f"MFIV_{H}"].astype(float)

    features["garch"] = X_df["garch"].astype(float)
    features["rt"] = X_df["rt"].astype(float)

    rv = X_df["rv"].astype(float)

    for H in RV_HORIZONS:
        features[f"rv_past_{H}"] = rv.rolling(H, min_periods=H).sum().shift(1)

    for H1, H2 in zip(MFIV_HORIZONS[:-1], MFIV_HORIZONS[1:]):
        features[f"mfiv_slope_{H1}_{H2}"] = (
            features[f"MFIV_{H2}"] / H2
            - features[f"MFIV_{H1}"] / H1
        )

    features_by_year[y] = features.replace([np.inf, -np.inf], np.nan)


FEATURE_COLS = list(features_by_year[2019].columns)

features_all = pd.concat([
    features_by_year[y].assign(year=y)
    for y in YEARS
]).sort_index()

print(FEATURE_COLS)
print("n features:", len(FEATURE_COLS))

['MFIV_12', 'MFIV_24', 'MFIV_72', 'MFIV_168', 'MFIV_336', 'garch', 'rt', 'rv_past_12', 'rv_past_24', 'rv_past_72', 'mfiv_slope_12_24', 'mfiv_slope_24_72', 'mfiv_slope_72_168', 'mfiv_slope_168_336']
n features: 14


In [ ]:
# =============================================================================================================
# ======================================= FEATURE_DATA_INTEGRITY ==============================================
# =============================================================================================================

rows = []

for y in YEARS:
    x = features_by_year[y]

    full_index = pd.date_range(
        x.index.min(),
        x.index.max(),
        freq="h",
        tz="UTC"
    )

    rows.append({
        "year": y,
        "rows": len(x),
        "duplicate_times": x.index.duplicated().sum(),
        "missing_hours": len(full_index.difference(x.index)),
        "rows_with_nan": x.isna().any(axis=1).sum(),
        "complete_rows": x.dropna().shape[0]
    })

integrity_check = pd.DataFrame(rows)

print(integrity_check.to_string(index=False))


missing_by_feature = pd.DataFrame({
    y: features_by_year[y].isna().sum()
    for y in YEARS
})

print("\nMISSING BY FEATURE")
print(missing_by_feature)

 year  rows  duplicate_times  missing_hours  rows_with_nan  complete_rows
 2019  8760                0              0             72           8688
 2020  8784                0              0             72           8712
 2021  8760                0              0             72           8688

MISSING BY FEATURE
                    2019  2020  2021
MFIV_12                0     0     0
MFIV_24                0     0     0
MFIV_72                0     0     0
MFIV_168               0     0     0
MFIV_336               0     0     0
garch                 10    10    10
rt                     0     0     0
rv_past_12            12    12    12
rv_past_24            24    24    24
rv_past_72            72    72    72
mfiv_slope_12_24       0     0     0
mfiv_slope_24_72       0     0     0
mfiv_slope_72_168      0     0     0
mfiv_slope_168_336     0     0     0


In [ ]:
# =============================================================================================================
# ======================================= FEATURE_DESCRIPTIVE_STATISTICS =====================================
# =============================================================================================================

descriptive_stats = features_all[FEATURE_COLS].describe(
    percentiles=[.01, .05, .25, .50, .75, .95, .99]
).T

print(descriptive_stats)

                      count      mean       std           min        1%  \
MFIV_12             26304.0  0.000986  0.000831  1.002077e-04  0.000154   
MFIV_24             26304.0  0.001925  0.001551  2.004155e-04  0.000308   
MFIV_72             26304.0  0.005681  0.004348  5.935267e-04  0.000975   
MFIV_168            26304.0  0.012734  0.009111  1.384896e-03  0.002376   
MFIV_336            26304.0  0.024533  0.016839  2.355673e-03  0.004666   
garch               26274.0  0.000091  0.003312  1.444463e-09  0.000001   
rt                  26304.0  0.000096  0.008618 -2.455685e-01 -0.023855   
rv_past_12          26268.0  0.000892  0.002963  5.546915e-06  0.000019   
rv_past_24          26232.0  0.001785  0.005060  1.948254e-05  0.000052   
rv_past_72          26088.0  0.005361  0.010835  9.830630e-05  0.000298   
mfiv_slope_12_24    26304.0 -0.000002  0.000020 -7.282651e-04 -0.000087   
mfiv_slope_24_72    26304.0 -0.000001  0.000022 -4.837129e-04 -0.000073   
mfiv_slope_72_168   26304

In [ ]:
# =============================================================================================================
# ======================================= FEATURE_STATIONARITY ================================================
# =============================================================================================================

rows = []

for y in YEARS:
    for col in FEATURE_COLS:
        x = features_by_year[y][col].dropna()

        adf = adfuller(x, autolag="AIC")
        kpss_test = kpss(x, regression="c", nlags="auto")

        rows.append({
            "year": y,
            "feature": col,
            "n": len(x),
            "adf_stat": adf[0],
            "adf_p": adf[1],
            "kpss_stat": kpss_test[0],
            "kpss_p": kpss_test[1],
            "adf_stationary": adf[1] < 0.05,
            "kpss_stationary": kpss_test[1] >= 0.05
        })

stationarity_check = pd.DataFrame(rows)

stationarity_check["both_support_stationary"] = (
    stationarity_check["adf_stationary"]
    & stationarity_check["kpss_stationary"]
)

print(stationarity_check.to_string(index=False))

 year            feature    n   adf_stat        adf_p  kpss_stat   kpss_p  adf_stationary  kpss_stationary  both_support_stationary
 2019            MFIV_12 8760  -4.907686 3.369275e-05   2.127075 0.010000            True            False                    False
 2019            MFIV_24 8760  -5.030252 1.932865e-05   2.175671 0.010000            True            False                    False
 2019            MFIV_72 8760  -4.188952 6.878540e-04   2.260854 0.010000            True            False                    False
 2019           MFIV_168 8760  -3.418276 1.035139e-02   2.403537 0.010000            True            False                    False
 2019           MFIV_336 8760  -3.825734 2.654725e-03   2.500267 0.010000            True            False                    False
 2019              garch 8750 -14.613534 3.987596e-27   0.846632 0.010000            True            False                    False
 2019                 rt 8760 -15.307332 4.232461e-28   0.338881 0.100000   

In [ ]:
# =============================================================================================================
# ======================================= FEATURE_AUTOCORRELATION =============================================
# =============================================================================================================

LAGS = [1, 12, 24, 72, 168]

rows = []

for y in YEARS:
    for col in FEATURE_COLS:
        x = features_by_year[y][col].dropna()

        row = {
            "year": y,
            "feature": col
        }

        for lag in LAGS:
            row[f"acf_{lag}"] = x.autocorr(lag=lag)

        rows.append(row)

autocorrelation_check = pd.DataFrame(rows)

print(autocorrelation_check.to_string(index=False))

 year            feature     acf_1    acf_12    acf_24    acf_72   acf_168
 2019            MFIV_12  0.929120  0.787709  0.722072  0.624371  0.519119
 2019            MFIV_24  0.926429  0.807093  0.746569  0.647222  0.548524
 2019            MFIV_72  0.926645  0.817119  0.768052  0.686985  0.588283
 2019           MFIV_168  0.913954  0.808735  0.770133  0.711649  0.621135
 2019           MFIV_336  0.944672  0.758170  0.712392  0.677746  0.602407
 2019              garch  0.621239  0.102100  0.104335  0.071675  0.026590
 2019                 rt -0.036481  0.011871 -0.041906  0.022376 -0.007084
 2019         rv_past_12  0.944486  0.264730  0.207329  0.128545  0.057870
 2019         rv_past_24  0.978305  0.686609  0.328611  0.195087  0.095626
 2019         rv_past_72  0.995439  0.928368  0.832675  0.414366  0.254053
 2019   mfiv_slope_12_24  0.915592  0.063518 -0.004906 -0.005391 -0.005451
 2019   mfiv_slope_24_72  0.745481  0.264153  0.168378 -0.018086  0.040006
 2019  mfiv_slope_72_168 

In [ ]:
# =============================================================================================================
# ======================================= LJUNG BOX TEST ======================================================
# =============================================================================================================

LB_LAGS = [12, 24, 72, 168]

rows = []

for y in YEARS:
    for col in FEATURE_COLS:
        x = features_by_year[y][col].dropna()

        lb = acorr_ljungbox(
            x,
            lags=LB_LAGS,
            return_df=True
        )

        for lag in LB_LAGS:
            rows.append({
                "year": y,
                "feature": col,
                "lag": lag,
                "lb_stat": lb.loc[lag, "lb_stat"],
                "lb_p": lb.loc[lag, "lb_pvalue"]
            })

ljung_box_check = pd.DataFrame(rows)

print(ljung_box_check.to_string(index=False))

 year            feature  lag       lb_stat         lb_p
 2019            MFIV_12   12  74742.262170 0.000000e+00
 2019            MFIV_12   24 132697.169705 0.000000e+00
 2019            MFIV_12   72 317816.218392 0.000000e+00
 2019            MFIV_12  168 598908.226754 0.000000e+00
 2019            MFIV_24   12  75905.244558 0.000000e+00
 2019            MFIV_24   24 137500.863238 0.000000e+00
 2019            MFIV_24   72 336911.361254 0.000000e+00
 2019            MFIV_24  168 643274.972283 0.000000e+00
 2019            MFIV_72   12  77430.487619 0.000000e+00
 2019            MFIV_72   24 142050.031934 0.000000e+00
 2019            MFIV_72   72 361899.637987 0.000000e+00
 2019            MFIV_72  168 708225.125825 0.000000e+00
 2019           MFIV_168   12  75148.637906 0.000000e+00
 2019           MFIV_168   24 139684.864141 0.000000e+00
 2019           MFIV_168   72 371623.456295 0.000000e+00
 2019           MFIV_168  168 756074.015370 0.000000e+00
 2019           MFIV_336   12  

In [ ]:
# =============================================================================================================
# ======================================= FEATURE_CORRELATION =================================================
# =============================================================================================================

complete_features = features_all[FEATURE_COLS].dropna()

pearson_corr = complete_features.corr(method="pearson")
spearman_corr = complete_features.corr(method="spearman")

print("\nPEARSON CORRELATION")
print(pearson_corr)

print("\nSPEARMAN CORRELATION")
print(spearman_corr)


rows = []

for i, col1 in enumerate(FEATURE_COLS):
    for col2 in FEATURE_COLS[i + 1:]:
        corr = pearson_corr.loc[col1, col2]

        if abs(corr) >= 0.90:
            rows.append({
                "feature_1": col1,
                "feature_2": col2,
                "pearson_corr": corr
            })

high_correlation_pairs = pd.DataFrame(rows)

print("\nHIGH CORRELATION PAIRS")
print(high_correlation_pairs.to_string(index=False))


PEARSON CORRELATION
                     MFIV_12   MFIV_24   MFIV_72  MFIV_168  MFIV_336  \
MFIV_12             1.000000  0.956260  0.898403  0.857890  0.829134   
MFIV_24             0.956260  1.000000  0.939124  0.889492  0.865471   
MFIV_72             0.898403  0.939124  1.000000  0.954619  0.922669   
MFIV_168            0.857890  0.889492  0.954619  1.000000  0.962519   
MFIV_336            0.829134  0.865471  0.922669  0.962519  1.000000   
garch               0.032111  0.032752  0.024096  0.020574  0.018763   
rt                 -0.014890 -0.014302 -0.010045  0.000294 -0.001647   
rv_past_12          0.496836  0.500416  0.385129  0.348952  0.324336   
rv_past_24          0.560219  0.555563  0.446296  0.423154  0.389308   
rv_past_72          0.597999  0.607698  0.582487  0.578977  0.546617   
mfiv_slope_12_24   -0.369525 -0.081549 -0.077796 -0.097424 -0.075754   
mfiv_slope_24_72   -0.343159 -0.359865 -0.017398  0.003801 -0.013055   
mfiv_slope_72_168  -0.414328 -0.454796 -0.4

In [ ]:
# =============================================================================================================
# ======================================= FEATURE_DISTRIBUTION ================================================
# =============================================================================================================

rows = []

for col in FEATURE_COLS:
    x = complete_features[col]
    jb = jarque_bera(x)

    rows.append({
        "feature": col,
        "n": len(x),
        "mean": x.mean(),
        "std": x.std(),
        "skew": x.skew(),
        "excess_kurtosis": x.kurt(),
        "jb_stat": jb.statistic,
        "jb_p": jb.pvalue
    })

distribution_check = pd.DataFrame(rows)

print(distribution_check.to_string(index=False))

           feature     n      mean      std       skew  excess_kurtosis      jb_stat  jb_p
           MFIV_12 26088  0.000984 0.000832   3.660311        24.031746 6.857641e+05   0.0
           MFIV_24 26088  0.001921 0.001551   3.369156        20.802639 5.195577e+05   0.0
           MFIV_72 26088  0.005667 0.004341   3.065799        17.835805 3.865133e+05   0.0
          MFIV_168 26088  0.012734 0.009127   3.016618        18.387460 4.069254e+05   0.0
          MFIV_336 26088  0.024539 0.016874   2.907497        18.738695 4.182839e+05   0.0
             garch 26088  0.000091 0.003324 159.726316     25699.690996 7.177711e+11   0.0
                rt 26088  0.000090 0.008609  -1.632404        72.182141 5.672932e+06   0.0
        rv_past_12 26088  0.000890 0.002967  19.205021       489.787594 2.622658e+08   0.0
        rv_past_24 26088  0.001781 0.005067  17.643458       426.389522 1.989027e+08   0.0
        rv_past_72 26088  0.005361 0.010835  10.566148       148.873820 2.456769e+07   0.0

In [ ]:
# =============================================================================================================
# ======================================= FEATURE_CROSS_YEAR_STABILITY =======================================
# =============================================================================================================

rows = []

for y in YEARS:
    for col in FEATURE_COLS:
        x = features_by_year[y][col].dropna()

        rows.append({
            "year": y,
            "feature": col,
            "n": len(x),
            "mean": x.mean(),
            "std": x.std(),
            "median": x.median(),
            "q25": x.quantile(.25),
            "q75": x.quantile(.75)
        })

cross_year_check = pd.DataFrame(rows)

print(cross_year_check.to_string(index=False))


mean_by_year = cross_year_check.pivot(
    index="feature",
    columns="year",
    values="mean"
)

std_by_year = cross_year_check.pivot(
    index="feature",
    columns="year",
    values="std"
)

print("\nMEAN BY YEAR")
print(mean_by_year)

print("\nSTD BY YEAR")
print(std_by_year)

 year            feature    n          mean      std        median           q25          q75
 2019            MFIV_12 8760  8.151164e-04 0.000586  6.247102e-04  4.675206e-04 1.084432e-03
 2019            MFIV_24 8760  1.621307e-03 0.001135  1.249420e-03  9.350412e-04 2.162521e-03
 2019            MFIV_72 8760  4.788253e-03 0.003194  3.750857e-03  2.805123e-03 6.460526e-03
 2019           MFIV_168 8760  1.072127e-02 0.006625  8.635284e-03  6.545288e-03 1.429757e-02
 2019           MFIV_336 8760  2.052094e-02 0.012856  1.656162e-02  1.244014e-02 2.670635e-02
 2019              garch 8750  5.791627e-05 0.000178  1.616987e-05  6.493872e-06 4.870604e-05
 2019                 rt 8760  7.598920e-05 0.007804  6.526519e-05 -2.258019e-03 2.407834e-03
 2019         rv_past_12 8748  7.316667e-04 0.001610  2.249497e-04  9.292512e-05 6.672437e-04
 2019         rv_past_24 8736  1.464966e-03 0.002561  6.082393e-04  2.429029e-04 1.411279e-03
 2019         rv_past_72 8688  4.409199e-03 0.005632  2.3183

In [ ]:
# =============================================================================================================
# ======================================= LOOKBACK_168_DIAGNOSTICS ============================================
# =============================================================================================================

SEQUENCE_LENGTH = 168

rows = []

for y in YEARS:
    x = features_by_year[y]
    valid_windows = 0

    for i in range(SEQUENCE_LENGTH - 1, len(x)):
        seq = x.iloc[i - SEQUENCE_LENGTH + 1:i + 1]

        if seq.index[-1] - seq.index[0] != pd.Timedelta(hours=SEQUENCE_LENGTH - 1):
            continue

        if seq.isna().any().any():
            continue

        valid_windows += 1

    possible_windows = max(len(x) - SEQUENCE_LENGTH + 1, 0)

    rows.append({
        "year": y,
        "possible_windows": possible_windows,
        "valid_168h_windows": valid_windows,
        "valid_fraction": valid_windows / possible_windows if possible_windows else np.nan
    })

lookback_window_check = pd.DataFrame(rows)

print(lookback_window_check.to_string(index=False))


lookback_dependence = autocorrelation_check[
    ["year", "feature", "acf_24", "acf_72", "acf_168"]
].copy()

print("\nLOOKBACK DEPENDENCE")
print(lookback_dependence.to_string(index=False))


adjacent_window_overlap = (SEQUENCE_LENGTH - 1) / SEQUENCE_LENGTH

print("\nADJACENT 168-HOUR WINDOW OVERLAP:", adjacent_window_overlap)

 year  possible_windows  valid_168h_windows  valid_fraction
 2019              8593                8521        0.991621
 2020              8617                8545        0.991644
 2021              8593                8521        0.991621

LOOKBACK DEPENDENCE
 year            feature    acf_24    acf_72   acf_168
 2019            MFIV_12  0.722072  0.624371  0.519119
 2019            MFIV_24  0.746569  0.647222  0.548524
 2019            MFIV_72  0.768052  0.686985  0.588283
 2019           MFIV_168  0.770133  0.711649  0.621135
 2019           MFIV_336  0.712392  0.677746  0.602407
 2019              garch  0.104335  0.071675  0.026590
 2019                 rt -0.041906  0.022376 -0.007084
 2019         rv_past_12  0.207329  0.128545  0.057870
 2019         rv_past_24  0.328611  0.195087  0.095626
 2019         rv_past_72  0.832675  0.414366  0.254053
 2019   mfiv_slope_12_24 -0.004906 -0.005391 -0.005451
 2019   mfiv_slope_24_72  0.168378 -0.018086  0.040006
 2019  mfiv_slope_72_168

In [ ]:
# PRE MODELLING TESTS
# SEARCH WORDS :
# FEATURE_DATA_INTEGRITY
# FEATURE_DESCRIPTIVE_STATISTICS
# FEATURE_STATIONARITY
# FEATURE_AUTOCORRELATION
# FEATURE_CORRELATION
# FEATURE_DISTRIBUTION
# FEATURE_CROSS_YEAR_STABILITY
# LOOKBACK_168_DIAGNOSTICS

import pandas as pd
import numpy as np
import warnings
import os, sys
import pickle

from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.stats.diagnostic import acorr_ljungbox
from scipy.stats import jarque_bera


warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)


sys.path.insert(0,"/Users/shah/CODE_BOOK_4/THESIS_WORKING/THESIS_2")
sys.path.insert(0,"/Users/shah/CODE_BOOK_4/code_document/REPOSITORIES/MFIV_calibration")

os.chdir("/Users/shah/CODE_BOOK_4/THESIS_WORKING/THESIS_2")


# =============================================================================================================
# ======================================= LOAD BASE DATA ======================================================
# =============================================================================================================

with open("/Users/shah/CODE_BOOK_4/code_document/REPOSITORIES/data/IS/outputs_by_year/outputs_by_year_03.pkl","rb") as f:
    outputs_by_year = pickle.load(f)

with open("/Users/shah/CODE_BOOK_4/code_document/REPOSITORIES/data/IS/x_by_year_fut/x_by_year.pkl","rb") as f:
    X_by_year = pd.read_pickle(f)


YEARS = [2019, 2020, 2021]
QUARTERS = ["q1", "q2", "q3", "q4"]


# =============================================================================================================
# ======================================= PREPARE X_BY_YEAR ===================================================
# =============================================================================================================

for y in YEARS:
    X_by_year[y] = X_by_year[y].copy()
    X_by_year[y]["rv"] = X_by_year[y]["rt"] ** 2


Xq_by_year = {}

for y in YEARS:
    Xq_by_year[y] = {}
    Xq_by_year[y]["q1"] = X_by_year[y].loc[f"{y}-01-01" : f"{y}-03-31 23:59:59"]
    Xq_by_year[y]["q2"] = X_by_year[y].loc[f"{y}-04-01" : f"{y}-06-30 23:59:59"]
    Xq_by_year[y]["q3"] = X_by_year[y].loc[f"{y}-07-01" : f"{y}-09-30 23:59:59"]
    Xq_by_year[y]["q4"] = X_by_year[y].loc[f"{y}-10-01" : f"{y}-12-31 23:59:59"]


# =============================================================================================================
# ======================================= MFIV SCALING ========================================================
# =============================================================================================================
# CONVERTS ANNUALIZED MFIV -> CUMULATIVE H-HOUR DECIMAL VARIANCE

MFIVS = list(range(2, 577))

for y in YEARS:
    for q in QUARTERS:
        for H in MFIVS:
            Xq_by_year[y][q][f"MFIV_{H}"] *= (H / 8760.0)

for y in YEARS:
    for q in QUARTERS:
        Xq_by_year[y][q] = Xq_by_year[y][q].loc[:, ~Xq_by_year[y][q].columns.duplicated(keep="first")]


# =============================================================================================================
# ======================================= BUILD COMMON MODEL FEATURES ========================================
# =============================================================================================================

MFIV_HORIZONS = [12, 24, 72, 168, 336]
RV_HORIZONS = [12, 24, 72]

features_by_year = {}

for y in YEARS:
    X_df = pd.concat([Xq_by_year[y][q] for q in QUARTERS]).sort_index().copy()
    garch_df = pd.concat([outputs_by_year[y][q]["garch"] for q in QUARTERS]).sort_index()

    X_df.index = pd.to_datetime(X_df.index, utc=True).floor("h")
    X_df = X_df.loc[~X_df.index.duplicated(keep="first")]
    X_df["garch"] = garch_df.reindex(X_df.index)

    features = pd.DataFrame(index=X_df.index)

    for H in MFIV_HORIZONS:
        features[f"MFIV_{H}"] = X_df[f"MFIV_{H}"].astype(float)

    features["garch"] = X_df["garch"].astype(float)
    features["rt"] = X_df["rt"].astype(float)

    rv = X_df["rv"].astype(float)

    for H in RV_HORIZONS:
        features[f"rv_past_{H}"] = rv.rolling(H, min_periods=H).sum().shift(1)

    for H1, H2 in zip(MFIV_HORIZONS[:-1], MFIV_HORIZONS[1:]):
        features[f"mfiv_slope_{H1}_{H2}"] = (
            features[f"MFIV_{H2}"] / H2
            - features[f"MFIV_{H1}"] / H1
        )

    features_by_year[y] = features.replace([np.inf, -np.inf], np.nan)


FEATURE_COLS = list(features_by_year[2019].columns)

features_all = pd.concat([
    features_by_year[y].assign(year=y)
    for y in YEARS
]).sort_index()

print(FEATURE_COLS)
print("n features:", len(FEATURE_COLS))


# =============================================================================================================
# ======================================= FEATURE_DATA_INTEGRITY ==============================================
# =============================================================================================================

rows = []

for y in YEARS:
    x = features_by_year[y]

    full_index = pd.date_range(
        x.index.min(),
        x.index.max(),
        freq="h",
        tz="UTC"
    )

    rows.append({
        "year": y,
        "rows": len(x),
        "duplicate_times": x.index.duplicated().sum(),
        "missing_hours": len(full_index.difference(x.index)),
        "rows_with_nan": x.isna().any(axis=1).sum(),
        "complete_rows": x.dropna().shape[0]
    })

integrity_check = pd.DataFrame(rows)

print(integrity_check.to_string(index=False))


missing_by_feature = pd.DataFrame({
    y: features_by_year[y].isna().sum()
    for y in YEARS
})

print("\nMISSING BY FEATURE")
print(missing_by_feature)


# =============================================================================================================
# ======================================= FEATURE_DESCRIPTIVE_STATISTICS =====================================
# =============================================================================================================

descriptive_stats = features_all[FEATURE_COLS].describe(
    percentiles=[.01, .05, .25, .50, .75, .95, .99]
).T

print(descriptive_stats)


# =============================================================================================================
# ======================================= FEATURE_STATIONARITY ================================================
# =============================================================================================================

rows = []

for y in YEARS:
    for col in FEATURE_COLS:
        x = features_by_year[y][col].dropna()

        adf = adfuller(x, autolag="AIC")
        kpss_test = kpss(x, regression="c", nlags="auto")

        rows.append({
            "year": y,
            "feature": col,
            "n": len(x),
            "adf_stat": adf[0],
            "adf_p": adf[1],
            "kpss_stat": kpss_test[0],
            "kpss_p": kpss_test[1],
            "adf_stationary": adf[1] < 0.05,
            "kpss_stationary": kpss_test[1] >= 0.05
        })

stationarity_check = pd.DataFrame(rows)

stationarity_check["both_support_stationary"] = (
    stationarity_check["adf_stationary"]
    & stationarity_check["kpss_stationary"]
)

print(stationarity_check.to_string(index=False))


# =============================================================================================================
# ======================================= FEATURE_AUTOCORRELATION =============================================
# =============================================================================================================

LAGS = [1, 12, 24, 72, 168]

rows = []

for y in YEARS:
    for col in FEATURE_COLS:
        x = features_by_year[y][col].dropna()

        row = {
            "year": y,
            "feature": col
        }

        for lag in LAGS:
            row[f"acf_{lag}"] = x.autocorr(lag=lag)

        rows.append(row)

autocorrelation_check = pd.DataFrame(rows)

print(autocorrelation_check.to_string(index=False))


# =============================================================================================================
# ======================================= LJUNG BOX TEST ======================================================
# =============================================================================================================

LB_LAGS = [12, 24, 72, 168]

rows = []

for y in YEARS:
    for col in FEATURE_COLS:
        x = features_by_year[y][col].dropna()

        lb = acorr_ljungbox(
            x,
            lags=LB_LAGS,
            return_df=True
        )

        for lag in LB_LAGS:
            rows.append({
                "year": y,
                "feature": col,
                "lag": lag,
                "lb_stat": lb.loc[lag, "lb_stat"],
                "lb_p": lb.loc[lag, "lb_pvalue"]
            })

ljung_box_check = pd.DataFrame(rows)

print(ljung_box_check.to_string(index=False))


# =============================================================================================================
# ======================================= FEATURE_CORRELATION =================================================
# =============================================================================================================

complete_features = features_all[FEATURE_COLS].dropna()

pearson_corr = complete_features.corr(method="pearson")
spearman_corr = complete_features.corr(method="spearman")

print("\nPEARSON CORRELATION")
print(pearson_corr)

print("\nSPEARMAN CORRELATION")
print(spearman_corr)


rows = []

for i, col1 in enumerate(FEATURE_COLS):
    for col2 in FEATURE_COLS[i + 1:]:
        corr = pearson_corr.loc[col1, col2]

        if abs(corr) >= 0.90:
            rows.append({
                "feature_1": col1,
                "feature_2": col2,
                "pearson_corr": corr
            })

high_correlation_pairs = pd.DataFrame(rows)

print("\nHIGH CORRELATION PAIRS")
print(high_correlation_pairs.to_string(index=False))


# =============================================================================================================
# ======================================= FEATURE_DISTRIBUTION ================================================
# =============================================================================================================

rows = []

for col in FEATURE_COLS:
    x = complete_features[col]
    jb = jarque_bera(x)

    rows.append({
        "feature": col,
        "n": len(x),
        "mean": x.mean(),
        "std": x.std(),
        "skew": x.skew(),
        "excess_kurtosis": x.kurt(),
        "jb_stat": jb.statistic,
        "jb_p": jb.pvalue
    })

distribution_check = pd.DataFrame(rows)

print(distribution_check.to_string(index=False))


# =============================================================================================================
# ======================================= FEATURE_CROSS_YEAR_STABILITY =======================================
# =============================================================================================================

rows = []

for y in YEARS:
    for col in FEATURE_COLS:
        x = features_by_year[y][col].dropna()

        rows.append({
            "year": y,
            "feature": col,
            "n": len(x),
            "mean": x.mean(),
            "std": x.std(),
            "median": x.median(),
            "q25": x.quantile(.25),
            "q75": x.quantile(.75)
        })

cross_year_check = pd.DataFrame(rows)

print(cross_year_check.to_string(index=False))


mean_by_year = cross_year_check.pivot(
    index="feature",
    columns="year",
    values="mean"
)

std_by_year = cross_year_check.pivot(
    index="feature",
    columns="year",
    values="std"
)

print("\nMEAN BY YEAR")
print(mean_by_year)

print("\nSTD BY YEAR")
print(std_by_year)


# =============================================================================================================
# ======================================= LOOKBACK_168_DIAGNOSTICS ============================================
# =============================================================================================================

SEQUENCE_LENGTH = 168

rows = []

for y in YEARS:
    x = features_by_year[y]
    valid_windows = 0

    for i in range(SEQUENCE_LENGTH - 1, len(x)):
        seq = x.iloc[i - SEQUENCE_LENGTH + 1:i + 1]

        if seq.index[-1] - seq.index[0] != pd.Timedelta(hours=SEQUENCE_LENGTH - 1):
            continue

        if seq.isna().any().any():
            continue

        valid_windows += 1

    possible_windows = max(len(x) - SEQUENCE_LENGTH + 1, 0)

    rows.append({
        "year": y,
        "possible_windows": possible_windows,
        "valid_168h_windows": valid_windows,
        "valid_fraction": valid_windows / possible_windows if possible_windows else np.nan
    })

lookback_window_check = pd.DataFrame(rows)

print(lookback_window_check.to_string(index=False))


lookback_dependence = autocorrelation_check[
    ["year", "feature", "acf_24", "acf_72", "acf_168"]
].copy()

print("\nLOOKBACK DEPENDENCE")
print(lookback_dependence.to_string(index=False))


adjacent_window_overlap = (SEQUENCE_LENGTH - 1) / SEQUENCE_LENGTH

print("\nADJACENT 168-HOUR WINDOW OVERLAP:", adjacent_window_overlap)


# =============================================================================================================
# ======================================= CHECKPOINT 1 ========================================================
# =============================================================================================================
# WE NOW HAVE :
# integrity_check
# missing_by_feature
# descriptive_stats
# stationarity_check
# autocorrelation_check
# ljung_box_check
# pearson_corr
# spearman_corr
# high_correlation_pairs
# distribution_check
# cross_year_check
# mean_by_year
# std_by_year
# lookback_window_check
# lookback_dependence

['MFIV_12', 'MFIV_24', 'MFIV_72', 'MFIV_168', 'MFIV_336', 'garch', 'rt', 'rv_past_12', 'rv_past_24', 'rv_past_72', 'mfiv_slope_12_24', 'mfiv_slope_24_72', 'mfiv_slope_72_168', 'mfiv_slope_168_336']
n features: 14
 year  rows  duplicate_times  missing_hours  rows_with_nan  complete_rows
 2019  8760                0              0             72           8688
 2020  8784                0              0             72           8712
 2021  8760                0              0             72           8688

MISSING BY FEATURE
                    2019  2020  2021
MFIV_12                0     0     0
MFIV_24                0     0     0
MFIV_72                0     0     0
MFIV_168               0     0     0
MFIV_336               0     0     0
garch                 10    10    10
rt                     0     0     0
rv_past_12            12    12    12
rv_past_24            24    24    24
rv_past_72            72    72    72
mfiv_slope_12_24       0     0     0
mfiv_slope_24_72       0   

In [ ]:
# =============================================================================================================
# ======================================= FEATURE_OUTLIER_DIAGNOSTICS =========================================
# =============================================================================================================

rows = []

for y in YEARS:
    for col in FEATURE_COLS:
        x = features_by_year[y][col].dropna()

        q25 = x.quantile(.25)
        q75 = x.quantile(.75)
        robust_std = (q75 - q25) / 1.349
        z = (x - x.mean()) / x.std()

        rows.append({
            "year": y,
            "feature": col,
            "median": x.median(),
            "q95": x.quantile(.95),
            "q99": x.quantile(.99),
            "q999": x.quantile(.999),
            "min": x.min(),
            "max": x.max(),
            "std": x.std(),
            "robust_std": robust_std,
            "std_to_robust": x.std() / robust_std if robust_std > 0 else np.nan,
            "max_abs_z": z.abs().max()
        })

outlier_check = pd.DataFrame(rows)

print(outlier_check.to_string(index=False))

 year            feature        median      q95      q99     q999           min      max      std   robust_std  std_to_robust  max_abs_z
 2019            MFIV_12  6.247102e-04 0.001862 0.002760 0.004948  1.042632e-04 0.013562 0.000586 4.573103e-04   1.282229e+00  21.737652
 2019            MFIV_24  1.249420e-03 0.003714 0.005381 0.008882  2.085264e-04 0.027123 0.001135 9.099184e-04   1.247333e+00  22.469125
 2019            MFIV_72  3.750857e-03 0.010943 0.014400 0.023567  6.255793e-04 0.081369 0.003194 2.709713e-03   1.178631e+00  23.978413
 2019           MFIV_168  8.635284e-03 0.023189 0.030889 0.037185  1.459685e-03 0.189862 0.006625 5.746688e-03   1.152866e+00  27.039425
 2019           MFIV_336  1.656162e-02 0.043732 0.059860 0.071135  2.355673e-03 0.379724 0.012856 1.057540e-02   1.215674e+00  27.939952
 2019              garch  1.616987e-05 0.000231 0.000631 0.002044  1.150583e-07 0.008541 0.000178 3.129145e-05   5.673103e+00  47.786597
 2019                 rt  6.526519e-05 0.

In [ ]:
# =============================================================================================================
# ======================================= TARGET_DISTRIBUTION_DIAGNOSTICS =====================================
# =============================================================================================================

from pathlib import Path
from srcB.tcn_model_v2 import build_forward_targets, make_training_sequences, TARGET_COLS


DATA_DIR = Path("../data/rawData")

d_btc_raw_by_year = {}
df_raw_by_year = {}

for y in YEARS:
    d_btc_raw_by_year[y] = pd.read_parquet(DATA_DIR/f"btc_underlying_minutely_{y}.parquet")
    df_raw_by_year[y] = pd.read_parquet(DATA_DIR/f"derivative_data_{y}.parquet")

for y in YEARS:
    d_btc_raw_by_year[y] = d_btc_raw_by_year[y].set_index(
        pd.to_datetime(d_btc_raw_by_year[y]["date_time"])
    )
    df_raw_by_year[y] = df_raw_by_year[y].set_index(
        pd.to_datetime(df_raw_by_year[y]["date_time"])
    )


with open("/Users/shah/CODE_BOOK_4/code_document/REPOSITORIES/data/IS/ready_data/d_btc_by_year_all.pkl","rb") as f:
    d_btc_by_year = pickle.load(f)

for y in YEARS:
    d_btc_raw_by_year[y]["rfr"] = d_btc_by_year[y]["rfr"].reindex(
        d_btc_raw_by_year[y].index,
        method="ffill"
    )


targets_by_year = {}
model_targets_by_year = {}

for y in YEARS:
    targets = build_forward_targets(
        option_df=df_raw_by_year[y],
        time_df=d_btc_raw_by_year[y],
        times=features_by_year[y].dropna().index,
        forward_hours=24,
        delta_target=.20
    )

    _, y_array, idx = make_training_sequences(
        features_by_year[y],
        targets,
        sequence_length=168
    )

    targets_by_year[y] = targets

    model_targets_by_year[y] = pd.DataFrame(
        y_array,
        index=idx,
        columns=TARGET_COLS
    )


rows = []

for y in YEARS:
    for col in TARGET_COLS:
        x = model_targets_by_year[y][col].dropna()

        rows.append({
            "year": y,
            "target": col,
            "n": len(x),
            "mean": x.mean(),
            "std": x.std(),
            "min": x.min(),
            "q01": x.quantile(.01),
            "q05": x.quantile(.05),
            "q25": x.quantile(.25),
            "median": x.median(),
            "q75": x.quantile(.75),
            "q95": x.quantile(.95),
            "q99": x.quantile(.99),
            "max": x.max(),
            "skew": x.skew(),
            "excess_kurtosis": x.kurt()
        })

target_distribution_check = pd.DataFrame(rows)

print(target_distribution_check.to_string(index=False))

 year                target    n      mean      std       min       q01       q05       q25    median       q75      q95      q99      max      skew  excess_kurtosis
 2019  target_long_straddle 2196 -0.003558 0.031705 -0.120600 -0.058704 -0.036027 -0.018421 -0.009770  0.001480 0.056505 0.117078 0.281746  2.470130        12.104155
 2019 target_short_straddle 2196 -0.004635 0.038653 -1.046254 -0.128516 -0.062446 -0.009984  0.001345  0.010246 0.030699 0.058931 0.246220 -9.924800       244.115509
 2019  target_long_strangle 2196 -0.002338 0.019148 -0.051115 -0.025709 -0.018654 -0.010705 -0.006200 -0.001195 0.030883 0.080113 0.223377  4.462492        31.618628
 2019 target_short_strangle 2196 -0.003220 0.019630 -0.174096 -0.092251 -0.036395 -0.004306  0.001005  0.005373 0.012362 0.019109 0.115386 -4.039989        25.907949
 2020  target_long_straddle 4109 -0.001548 0.035144 -0.333320 -0.057646 -0.032386 -0.015556 -0.007788  0.003827 0.049841 0.139237 0.430000  3.211587        33.757366
 202

In [ ]:
# =============================================================================================================
# ======================================= GARCH EXTREME OBSERVATIONS ==========================================
# =============================================================================================================

garch_extremes = []

for y in YEARS:
    x = features_by_year[y].copy()

    top = x.nlargest(15, "garch")[
        ["garch", "rt", "rv_past_12", "rv_past_24", "rv_past_72",
         "MFIV_12", "MFIV_24", "MFIV_72"]
    ].copy()

    top["year"] = y
    garch_extremes.append(top)

garch_extremes = pd.concat(garch_extremes).sort_values("garch", ascending=False)

print(garch_extremes.to_string())

                              garch        rt  rv_past_12  rv_past_24  rv_past_72   MFIV_12   MFIV_24   MFIV_72  year
date_time                                                                                                            
2020-04-27 00:00:00+00:00  0.534917  0.018976    0.000357    0.000511    0.001467  0.000631  0.001263  0.004076  2020
2020-03-13 04:00:00+00:00  0.012546  0.029318    0.088832    0.151069    0.155761  0.008728  0.017456  0.019814  2020
2020-03-13 05:00:00+00:00  0.012285 -0.047223    0.089598    0.151907    0.156591  0.008974  0.017948  0.020172  2020
2020-03-13 06:00:00+00:00  0.010558  0.042983    0.091822    0.154111    0.158819  0.009227  0.018455  0.020537  2020
2020-03-13 07:00:00+00:00  0.009551  0.055430    0.093660    0.155946    0.160666  0.006682  0.013363  0.018150  2020
2020-03-16 00:00:00+00:00  0.009360  0.001970    0.019040    0.021054    0.135865  0.002483  0.004988  0.027276  2020
2021-05-21 16:00:00+00:00  0.008873 -0.026084    0.01188